# Push LoRA Adapters to HuggingFace Hub

Upload 6 LoRA adapters to private HuggingFace repos for inference.

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login(token='')

## Configuration

In [ ]:
import json
import os

from huggingface_hub import HfApi, upload_folder


# CONFIGURATION

HF_USERNAME = "Nicholas2k4"  

BASE_DIR = "./lora_output"

ADAPTERS = {
    "task_1_llama":      {"repo": "patent-task1-llama",   "base": "meta-llama/Llama-3.1-8B-Instruct"},
    "task_1_mistral":    {"repo": "patent-task1-mistral", "base": "mistralai/Mistral-7B-Instruct-v0.3"},
    "task_1_qwen":       {"repo": "patent-task1-qwen",    "base": "Qwen/Qwen3-8B"},
    "task_2_llama_8k":   {"repo": "patent-task2-llama",   "base": "meta-llama/Llama-3.1-8B-Instruct"},
    "task_2_mistral_8k": {"repo": "patent-task2-mistral", "base": "mistralai/Mistral-7B-Instruct-v0.3"},
    "task_2_qwen_8k":    {"repo": "patent-task2-qwen",    "base": "Qwen/Qwen3-8B"},
}

## Verify Adapter Files

In [ ]:
print(f"{'Folder':<25s} {'Status':<10s} {'Size':>10s}  Base Model")
print("─" * 90)

all_ok = True
for folder_name, info in ADAPTERS.items():
    adapter_path = os.path.join(BASE_DIR, folder_name, "final_adapter")
    config_path = os.path.join(adapter_path, "adapter_config.json")
    model_path = os.path.join(adapter_path, "adapter_model.safetensors")

    if not os.path.isdir(adapter_path):
        print(f"{folder_name:<25s} {'MISSING':<10s}")
        all_ok = False
        continue

    if not os.path.isfile(config_path) or not os.path.isfile(model_path):
        print(f"{folder_name:<25s} {'INCOMPLETE':<10s}")
        all_ok = False
        continue

    size_mb = sum(
        os.path.getsize(os.path.join(adapter_path, f))
        for f in os.listdir(adapter_path)
        if os.path.isfile(os.path.join(adapter_path, f))
    ) / (1024 * 1024)

    print(f"{folder_name:<25s} {'OK':<10s} {size_mb:>8.1f}MB  {info['base']}")

## Upload All Adapters

In [ ]:
api = HfApi()

try:
    user_info = api.whoami()
    print(f"Logged in as: {user_info['name']}")
except Exception:
    raise RuntimeError("Not logged in! Run notebook_login() above.")

results = []

for i, (folder_name, info) in enumerate(ADAPTERS.items(), 1):
    adapter_path = os.path.join(BASE_DIR, folder_name, "final_adapter")
    repo_id = f"{HF_USERNAME}/{info['repo']}"

    print(f"\n[{i}/6] {folder_name} → {repo_id}")

    if not os.path.isdir(adapter_path):
        print(f"Adapter not found")
        results.append((folder_name, repo_id, False))
        continue

    # Create private repo
    api.create_repo(repo_id=repo_id, repo_type="model", private=True, exist_ok=True)

    # Upload 
    try:
        upload_folder(
            folder_path=adapter_path,
            repo_id=repo_id,
            repo_type="model",
            commit_message=f"Upload LoRA adapter: {folder_name}",
            ignore_patterns=[".DS_Store", "training_args.bin", "README.md"],
        )
        print(f"Uploaded to https://huggingface.co/{repo_id}")
        results.append((folder_name, repo_id, True))
    except Exception as e:
        print(f"FAILED: {e}")
        results.append((folder_name, repo_id, False))